In [16]:
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from typing import TypedDict, Literal
from dotenv import load_dotenv
from pydantic import BaseModel, Field

In [17]:
load_dotenv()
model = ChatOpenAI(model="gpt-4o-mini")

In [18]:
class SentimentSchema(BaseModel):
    sentiment: Literal["positive", "negative"] = Field(description="The sentiment of the review")

sentiment_model = model.with_structured_output(SentimentSchema)

class DiagnosisSchema(BaseModel):
    issue_type: Literal["quality", "price", "delivery", "other"]
    tone: Literal["angry", "neutral", "fustrated", "disappointed"]
    urgency: Literal["low", "medium", "high"]
diagnosis_model = model.with_structured_output(DiagnosisSchema)


In [19]:
prompt = f"I really love this product! It's amazing!"
response = sentiment_model.invoke(prompt)
print(response)





sentiment='positive'


In [20]:
class ReviewState(TypedDict):
    review: str
    sentiment: Literal["positive", "negative"]
    diagnosis: dict
    response: str

In [21]:
def find_sentiment(state: ReviewState) :
    prompt = f"Find the sentiment of the following review: {state['review']}"
    response = sentiment_model.invoke(prompt)
    return {'sentiment': response.sentiment}

def check_sentiment(state: ReviewState) -> Literal["positive_response", "run_diagnosis"]:
    if state['sentiment'] == "positive":
        return "positive_response"
    else:
        return "run_diagnosis"

def positive_response(state: ReviewState) :
    prompt = f"Generate a response to the review: {state['review']}. Write a warm thank you message to the customer for their review."
    response = model.invoke(prompt).content
    return {'response': response}

def run_diagnosis(state: ReviewState) :
    prompt = f"diagnose this negative review: {state['review']}. Return issue type, tone and urgency."
    response = diagnosis_model.invoke(prompt)
    return {'diagnosis': response.model_dump()}

def negative_response(state: ReviewState) :
    diagnosis = state['diagnosis']
    prompt = f"you are a support assistant. the user had a {diagnosis['issue_type']} issue. the tone of the review is {diagnosis['tone']} and the urgency is {diagnosis['urgency']}. Write a response to the review."
    response = model.invoke(prompt).content
    return {'response': response}


In [22]:
graph = StateGraph(ReviewState)

graph.add_node('find_sentiment', find_sentiment)
graph.add_node('positive_response', positive_response)
graph.add_node('run_diagnosis', run_diagnosis)
graph.add_node('negative_response', negative_response)

graph.add_edge(START, 'find_sentiment')
graph.add_conditional_edges('find_sentiment', check_sentiment)
graph.add_edge('positive_response', END)
graph.add_edge('run_diagnosis', 'negative_response')
graph.add_edge('negative_response', END)

workflow = graph.compile()

In [24]:
initial_state = {"review": "This is the most stupid product I have ever seen on amazon. It broke after 2 days of use. I will never buy from this seller again."}

final_result = workflow.invoke(initial_state)
print(final_result)

{'review': 'This is the most stupid product I have ever seen on amazon. It broke after 2 days of use. I will never buy from this seller again.', 'sentiment': 'negative', 'diagnosis': {'issue_type': 'quality', 'tone': 'angry', 'urgency': 'high'}, 'response': "Subject: We’re Here to Help - Your Feedback Matters\n\nDear [User's Name],\n\nThank you for taking the time to share your feedback. I’m truly sorry to hear about the quality issue you’ve experienced, and I understand your frustration. Please know that we take these matters very seriously and are committed to resolving them as quickly as possible.\n\nTo assist you further, could you please provide more details about the issue? This will help us address your concerns more effectively. You can reach me directly at [your contact information] or reply to this message. \n\nWe appreciate your patience and understanding as we work to rectify this situation. Your satisfaction is our priority, and we are dedicated to making this right for yo